# **Correcciones post-EDA: outliers monetarios, mercado real y diseno de modelado**

Continuacion del EDA sobre el dataset limpio. Motivado por 3 hallazgos de la corrida anterior:

1. **Outlier absurdo**: una adjudicacion de ~2.1x10^17 COP (~140 veces el PIB anual de
   Colombia) via la entidad "EAG" destruye el HHI general (9,995), el Pareto
   ("1 proveedor = 80% del valor") y la serie de valor mensual.
2. **`adjudicado` estructuralmente vacio en modalidades no competitivas**: regimen
   especial y contratacion directa (82% del dataset) tienen 0.00% exacto — el campo
   no se llena para esas modalidades. El modelo de probabilidad de adjudicacion debe
   restringirse al universo competitivo.
3. **Fuga de informacion**: `respuestas_al_procedimiento` (corr 0.78 con adjudicado)
   se conoce solo DESPUES del cierre; no es feature valida al momento de publicacion.

Este notebook: (A) hace forense del outlier y define una regla de plausibilidad
monetaria, (B) rehace el analisis de mercado con valores creibles, (C) investiga los
picos de enero 2022 y enero 2026, (D) deja definidos los universos y features del
modelado para la capacidad 3.

In [1]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

DTYPE = {"nit_entidad": "string", "nit_del_proveedor_adjudicado": "string"}
df_lineas = pd.read_csv("secop_ctei_lineas_limpio.csv", dtype=DTYPE, low_memory=False)
df_proc = pd.read_csv("secop_ctei_procesos_limpio.csv", dtype={"nit_entidad": "string"}, low_memory=False)

for col in ["fecha_de_publicacion_del", "fecha_adjudicacion"]:
    for d in (df_lineas, df_proc):
        if col in d.columns:
            d[col] = pd.to_datetime(d[col], errors="coerce")

df_lineas["adjudicado_bool"] = df_lineas["adjudicado"].map({"Si": True, "No": False}).astype("boolean")
print(f"lineas: {len(df_lineas):,} | procesos: {len(df_proc):,}")

lineas: 495,284 | procesos: 492,797


# **A. Forense del outlier monetario y regla de plausibilidad**

## A.1 Ver las lineas mas grandes con contexto completo

Referencia de escala: el Presupuesto General de la Nacion es del orden de 5x10^14 COP
(~500 billones) al anio. Cualquier linea individual por encima de eso es fisicamente
imposible como contrato real.

In [2]:
top_lineas = df_lineas.nlargest(15, "valor_total_adjudicacion")[[
    "id_del_proceso", "entidad", "nombre_del_proveedor",
    "nombre_del_procedimiento", "modalidad_de_contratacion",
    "precio_base", "valor_total_adjudicacion", "fecha_de_publicacion_del", "urlproceso"
]]
top_lineas

,id_del_proceso,entidad,nombre_del_proveedor,nombre_del_procedimiento,modalidad_de_contratacion,precio_base,valor_total_adjudicacion,fecha_de_publicacion_del,urlproceso
281869,CO1.REQ.6849316,EAG,CORPORACION CONSTRULET S.A.S,REALIZAR CONSULTORIA PARA LA ELABORACIÓN DE ES...,Contratación régimen especial (con ofertas),463454668,214790230013979506,2024-09-16,https://community.secop.gov.co/Public/Tenderin...
431529,CO1.REQ.9483601,MINISTERIO DE MINAS Y ENERGIA,GECELCA S.A. E.S.P.,Administrar los recursos y gestionar proyectos...,Contratación Directa (con ofertas),4205027751839,4205027751839,2025-12-28,https://community.secop.gov.co/Public/Tenderin...
392211,CO1.REQ.8781496,PATRIMONIO AUTÓNOMO AEROCAFÉ,CONSORCIO AEROPUERTO DEL CAFE SK,CONVOCATORIA ABIERTA CONTRATACION OBRA LADO AI...,Contratación régimen especial (con ofertas),639583680291,634275135745,2025-08-25,https://community.secop.gov.co/Public/Tenderin...
475428,CO1.REQ.10324110,Empresa Distrital de Desarrollo y Renovación U...,CONSORCIO PLANTA CURVAL,INVITACION PUBLICA A OFERTAR DE MAYOR CUANTIA,Contratación régimen especial (con ofertas),861875326914,410758596204,2026-04-06,https://community.secop.gov.co/Public/Tenderin...
209227,CO1.REQ.5622810,MINISTERIO DE AGRICULTURA Y DESARROLLO RURAL,FONDO PARA EL FINANCIAMIENTO DEL SECTOR AGROPE...,CONTRATO INTERADMINISTRATIVO FINAGRO,Contratación Directa (con ofertas),392463605046,392463605046,2024-01-25,https://community.secop.gov.co/Public/Tenderin...
393873,CO1.REQ.8794819,DISTRITO ESPECIAL DE CIENCIA TECNOLOGIA E INNO...,EMPRESA DE DESARROLLO URBANO DE MEDELLIN,Contrato interadministrativo de mandato sin re...,Contratación Directa (con ofertas),379790887521,379790887521,2025-08-28,https://community.secop.gov.co/Public/Tenderin...
200636,CO1.REQ.5441709,AEROCIVIL,ENTerritorio S.A,Realizar la gerencia integral del proyecto par...,Contratación Directa (con ofertas),363763308569,363763308569,2023-12-20,https://community.secop.gov.co/Public/Tenderin...
260544,CO1.REQ.6474584,MUNICIPIO DE PEREIRA- OFICIAL,TEK SOLUCIONES TECNOLOGICAS S.A.S,SUMINISTRO DE LICENCIAS DE OFFICE PARA COLEGIO...,Selección abreviada subasta inversa,336502412,335128000000,2024-07-05,https://community.secop.gov.co/Public/Tenderin...
87374,CO1.REQ.3758486,FONDO FINANCIERO DISTRITAL DE SALUD..,"AGENCIA DISTRITAL PARA LA EDUCACIÓN SUPERIOR, ...",Fortalecimiento de las capacidades en salud de...,Contratación Directa (con ofertas),334710179630,334710179630,2022-12-23,https://community.secop.gov.co/Public/Tenderin...
427460,CO1.REQ.9409106,MINISTERIO DE MINAS Y ENERGIA,EMPRESA DISTRIBUIDORA DEL PACIFICO S.A. E.S.P,CE- MAICAO; ALBANIA Y MUNICIPIOS ALEDAÑOS,Contratación Directa (con ofertas),322709746890,322709746890,2025-12-10,https://community.secop.gov.co/Public/Tenderin...


In [3]:
# Cuantas lineas superan umbrales de plausibilidad crecientes
umbral_tabla = pd.DataFrame({
    "umbral_cop": [1e11, 1e12, 1e13, 1e14, 5e14],
    "descripcion": ["100 mil millones", "1 billon", "10 billones", "100 billones",
                     "~Presupuesto Gral. Nacion anual"],
})
umbral_tabla["lineas_que_superan"] = [
    (df_lineas["valor_total_adjudicacion"] > u).sum() for u in umbral_tabla["umbral_cop"]
]
umbral_tabla["valor_acumulado_de_esas_lineas"] = [
    df_lineas.loc[df_lineas["valor_total_adjudicacion"] > u, "valor_total_adjudicacion"].sum()
    for u in umbral_tabla["umbral_cop"]
]
umbral_tabla

,umbral_cop,descripcion,lineas_que_superan,valor_acumulado_de_esas_lineas
0,"100,000,000,000.00",100 mil millones,41,214802740890141180
1,"1,000,000,000,000.00",1 billon,2,214794435041731345
2,"10,000,000,000,000.00",10 billones,1,214790230013979506
3,"100,000,000,000,000.00",100 billones,1,214790230013979506
4,"500,000,000,000,000.00",~Presupuesto Gral. Nacion anual,1,214790230013979506


## A.2 Regla de plausibilidad (dos criterios combinados)

- **Absoluto**: valor de linea > 1x10^13 COP (10 billones) es implausible como
  adjudicacion individual en este universo (servicios de ingenieria/consultoria/
  educacion — no megaobras tipo metro, que ademas irian por otros segmentos UNSPSC).
- **Relativo**: valor > 100x el precio_base cuando precio_base > 1'000,000 COP
  (un precio base serio). Captura errores de digitos aun por debajo del umbral absoluto.

Las lineas marcadas NO se borran: se excluyen del analisis monetario con
`flag_valor_implausible`, documentando cuantas son y cuanto "valor" fantasma aportan.

In [4]:
UMBRAL_ABSOLUTO = 1e13
RATIO_MAX = 100

flag_abs = df_lineas["valor_total_adjudicacion"] > UMBRAL_ABSOLUTO
flag_rel = (
    (df_lineas["precio_base"] > 1e6)
    & (df_lineas["valor_total_adjudicacion"] > RATIO_MAX * df_lineas["precio_base"])
)
df_lineas["flag_valor_implausible"] = flag_abs | flag_rel

n_flag = df_lineas["flag_valor_implausible"].sum()
valor_fantasma = df_lineas.loc[df_lineas["flag_valor_implausible"], "valor_total_adjudicacion"].sum()
valor_total_bruto = df_lineas["valor_total_adjudicacion"].sum()

print(f"Lineas marcadas implausibles: {n_flag:,} "
      f"({n_flag / len(df_lineas):.3%} de las lineas)")
print(f"'Valor' que aportaban: {valor_fantasma:,.0f} COP "
      f"= {valor_fantasma / valor_total_bruto:.1%} del valor total bruto")
print()
print("Muestra de lineas marcadas:")
df_lineas[df_lineas["flag_valor_implausible"]].nlargest(10, "valor_total_adjudicacion")[[
    "entidad", "nombre_del_proveedor", "precio_base", "valor_total_adjudicacion"
]]

Lineas marcadas implausibles: 14 (0.003% de las lineas)
'Valor' que aportaban: 214,791,136,543,172,512 COP = 100.0% del valor total bruto

Muestra de lineas marcadas:


,entidad,nombre_del_proveedor,precio_base,valor_total_adjudicacion
281869,EAG,CORPORACION CONSTRULET S.A.S,463454668,214790230013979506
260544,MUNICIPIO DE PEREIRA- OFICIAL,TEK SOLUCIONES TECNOLOGICAS S.A.S,336502412,335128000000
362160,INSTITUTO FINANCIERO PARA EL DESARROLLO DEL VA...,C Y C SOLUCIONES INTEGRALES SAS,150000000,150000000000
489488,CVC,ALCALDIA MUNICIPAL DE GINEBRA,99070693,99070693000
273926,COMANDO GAULA MILITARES (COGAM),CENTROS RECREACIONALES Y SEDES HABITACIONALES ...,80000000,79957000000
288102,ALCALDIA DISTRITAL BARRANCABERMEJA,SEMPRO,68835000,68835000000
471819,ALCALDIA MUNICIPIO DE NOCAIMA,INCEGER,43200000,43200000000
180270,ESE HOSPITAL OCTAVIO OLIVARES,"SERVICIOS, SUMINISTROS Y MONTAJES S.A.S",43200000,43020000000
424510,ALCALDÍA MUNICIPAL DE SOPO,NETSOLUTIONS S.A.S,43048250,34000000000
38699,MUNICIPIO SANTA ROSA DE OSOS,RUIZ & ZAPATA S.A.S,18600000,18600000000


## A.3 Hallazgos — outliers

- **Solo 14 lineas (0.003% del total) explican el 100.0% del "valor" bruto adjudicado**
  segun la marca `flag_valor_implausible`. No es un factor de escala uniforme (no es
  "todo esta multiplicado por 10^7"): es un puñado de filas puntuales con montos
  fisicamente imposibles. El caso EAG (214.79 x10^15 COP) es, con enorme diferencia, el
  peor: su `precio_base` (463 millones) es coherente con una consultoria normal, pero
  `valor_total_adjudicacion` esta ~463,000 veces por encima del precio base — clarisimo
  error de captura/parseo del dato fuente, no un contrato real reescalado.
- El resto de las 14 lineas marcadas (Municipio de Pereira 335 mil millones, IFV
  150 mil millones, CVC 99 mil millones, COGAM 80 mil millones, etc.) son de una escala
  muchisimo mas chica que EAG pero igual implausibles frente a su `precio_base`
  (razon > 100x) — sugiere que el problema de captura de `valor_total_adjudicacion` no es
  exclusivo de una sola entidad/proveedor, sino un patron recurrente de baja frecuencia en
  la fuente (SECOP) que conviene seguir vigilando en futuras cargas del dataset.
- La regla combinada (absoluta 10^13 COP + relativa 100x precio_base) es razonable: es
  conservadora en el umbral absoluto (deja pasar contratos grandes pero fisicamente
  posibles, ej. Ministerio de Minas y Energia con 4.2 billones de precio_base) y agresiva
  en el umbral relativo para atrapar errores de digitos que no llegan al umbral absoluto.
  Las lineas se **excluyen del analisis monetario, no se corrigen/reescalan** — decision
  correcta dado que no hay forma confiable de inferir el valor real original.
- Pendiente para reportar: documentar estas 14 lineas (con `id_del_proceso` y `urlproceso`)
  como candidatas a validar manualmente contra el portal SECOP si hay tiempo, ya que
  siguen contando como "procesos adjudicados" validos en el conteo de Capacidad 1/3 —
  solo se excluyeron de los calculos *monetarios*, correctamente.

# **B. Analisis de mercado corregido (sin valores implausibles)**

In [5]:
df_mercado = df_lineas[
    (df_lineas["adjudicado_bool"] == True)
    & (df_lineas["valor_total_adjudicacion"] > 0)
    & (~df_lineas["flag_valor_implausible"])
].copy()
df_mercado["proveedor_id"] = df_mercado["nit_del_proveedor_adjudicado"].fillna(df_mercado["nombre_del_proveedor"])

def hhi(part): return (part ** 2).sum() * 10000

mercado = (
    df_mercado.groupby("proveedor_id")["valor_total_adjudicacion"].sum()
    .sort_values(ascending=False).reset_index()
)
mercado["participacion"] = mercado["valor_total_adjudicacion"] / mercado["valor_total_adjudicacion"].sum()
mercado["participacion_acum"] = mercado["participacion"].cumsum()

hhi_corregido = hhi(mercado["participacion"])
n_80 = (mercado["participacion_acum"] <= 0.80).sum() + 1
print(f"HHI corregido: {hhi_corregido:,.1f} "
      f"({'baja' if hhi_corregido < 1500 else 'moderada' if hhi_corregido < 2500 else 'alta'} concentracion)")
print(f"Proveedores que concentran el 80% del valor: {n_80:,} de {len(mercado):,} ({n_80/len(mercado):.1%})")

top20 = mercado.head(20).sort_values("valor_total_adjudicacion")
fig = go.Figure(go.Bar(x=top20["valor_total_adjudicacion"], y=top20["proveedor_id"].astype(str), orientation="h"))
fig.update_layout(title="Top 20 proveedores por valor adjudicado (corregido)",
                   xaxis_title="Valor adjudicado (COP)", template="plotly_white", height=600)
fig.show()

HHI corregido: 116.1 (baja concentracion)
Proveedores que concentran el 80% del valor: 1,044 de 17,179 (6.1%)


In [6]:
# Serie mensual de valor adjudicado corregida (la del EDA anterior estaba dominada por el outlier)
serie_valor = (
    df_mercado[df_mercado["fecha_de_publicacion_del"].notna()]
    .assign(anio_mes=lambda d: d["fecha_de_publicacion_del"].dt.to_period("M").dt.to_timestamp())
    .groupby("anio_mes")["valor_total_adjudicacion"].sum().reset_index()
)
fig = go.Figure(go.Scatter(x=serie_valor["anio_mes"], y=serie_valor["valor_total_adjudicacion"], mode="lines"))
fig.update_layout(title="Valor adjudicado por mes (sin implausibles)",
                   yaxis_title="COP", template="plotly_white")
fig.show()

In [7]:
# HHI por entidad, corregido, con las mismas condiciones (min 20 procesos adjudicados)
def hhi_por_entidad(df):
    filas = []
    for ent, sub in df.groupby("entidad"):
        val = sub.groupby("proveedor_id")["valor_total_adjudicacion"].sum()
        filas.append({"entidad": ent, "hhi": hhi(val / val.sum()),
                       "proveedores": sub["proveedor_id"].nunique(),
                       "procesos": sub["id_del_proceso"].nunique(),
                       "valor_total": val.sum()})
    return pd.DataFrame(filas)

hhi_ent = hhi_por_entidad(df_mercado)
print("Top 15 entidades por valor (corregido):")
display(hhi_ent.sort_values("valor_total", ascending=False).head(15))

filtradas = hhi_ent[hhi_ent["procesos"] >= 20].sort_values("hhi", ascending=False)
print("\nEntidades mas concentradas (min 20 procesos adjudicados):")
display(filtradas.head(10))
print("\nEntidades mas competitivas:")
display(filtradas.tail(10))

Top 15 entidades por valor (corregido):


,entidad,hhi,proveedores,procesos,valor_total
779,DISTRITO ESPECIAL DE CIENCIA TECNOLOGIA E INNO...,"1,383.75",264,949,5727377498700
1611,MINISTERIO DE MINAS Y ENERGIA,"7,610.81",68,87,4836052632118
1560,INVIAS,163.98,656,972,1631509329408
301,ANI,329.50,80,78,1316704294502
698,DEPARTAMENTO DE ANTIOQUIA//,"1,909.81",177,279,1238997511450
1607,MINISTERIO DE EDUCACION NACIONAL (MEN),"1,521.10",84,148,1092231185004
14,"AGENCIA DISTRITAL PARA LA EDUCACIÓN SUPERIOR, ...",714.58,69,54,1001244950027
1603,MINISTERIO DE AGRICULTURA Y DESARROLLO RURAL,"6,904.40",29,35,997418818827
2089,SECRETARIA DE EDUCACION DEL DISTRITO,"1,067.58",118,151,963028590183
6,AEROCIVIL,"1,996.42",203,241,836833475956



Entidades mas concentradas (min 20 procesos adjudicados):


,entidad,hhi,proveedores,procesos,valor_total
1083,FABRICA DE LICORES Y ALCOHOLES DE ANTIOQUIA,"9,150.34",37,50,103290566604
2125,SENA REGIONAL ANTIOQUIA Grupo de Apoyo Adminis...,"9,005.77",28,38,25791067148
1075,Empresa Distrital de Desarrollo y Renovación U...,"8,706.73",27,29,440396888176
59,ALCALDIA DE GIRARDOTA,"8,668.72",10,20,31932303879
1755,MUNICIPIO DE MARINILLA,"8,569.09",14,53,42172124813
1731,MUNICIPIO DE ITAGUI,"8,184.94",9,54,516973344747
1696,MUNICIPIO DE EL DOVIO,"8,019.47",2,22,327509705
1611,MINISTERIO DE MINAS Y ENERGIA,"7,610.81",68,87,4836052632118
2238,UAE SETP AVANTE PASTO,"7,590.42",17,23,19320053364
1673,MUNICIPIO DE CHIQUINQUIRA+,"7,546.07",21,33,46339928902



Entidades mas competitivas:


,entidad,hhi,proveedores,procesos,valor_total
345,ASOCIACION DE MUNICIPIOS CORPORACIÓN AGENCIA P...,403.96,70,160,675592325
2102,SECRETARIA DISTRITAL DEL HABITAT-,402.64,44,35,49062967835
1451,INSTITUTO DE DESARROLLO URBANO,400.15,120,137,665621756467
186,ALCALDIA MUNICIPIO DE ARAUCA,336.58,85,123,20115146824
301,ANI,329.50,80,78,1316704294502
71,ALCALDIA DE PASTO,317.81,71,107,8798583784
1165,GOBIERNO DEPARTAMENTAL,308.91,176,253,139169075198
1144,GOBERNACION DEL HUILA*,220.98,117,145,50612487385
851,EMPRESA DE DESARROLLO URBANO DE MEDELLIN,195.50,108,282,123014079676
1560,INVIAS,163.98,656,972,1631509329408


## B.1 Hallazgos — mercado corregido

- **El diagnostico de mercado cambia por completo al quitar el outlier**: HHI general
  pasa de 9,995.4 ("monopolio") a **116.1 ("baja concentracion")** — esta ultima cifra es
  la que hay que citar en el reporte. El "1 proveedor = 80% del valor" del EDA original
  era enteramente un artefacto: corregido, se necesitan **1,044 de 17,179 proveedores
  (6.1%)** para llegar al 80% del valor, un mercado CTeI razonablemente competido a nivel
  agregado nacional.
- EAG desaparece del top 15 de entidades por valor una vez corregido — confirma que su
  presencia anterior era 100% producto del outlier, no de actividad real relevante.
- **El panorama por entidad individual (no agregado) si mostraba concentracion real,
  independiente del outlier**: Ministerio de Minas y Energia (HHI 7,610.81, valor real
  4.8 billones) y Patrimonio Autonomo Aerocafe (HHI 9,391.22, pero solo 2 procesos/2
  proveedores — mas señal de bajo volumen que de mercado cerrado) ya aparecian
  concentrados en el EDA original y se mantienen igual tras la correccion (no dependian
  del outlier EAG). Nuevas entidades mas concentradas identificadas aqui: Fabrica de
  Licores y Alcoholes de Antioquia (9,150), SENA Regional Antioquia (9,006), Empresa
  Distrital de Desarrollo (8,707) — todas con relativamente pocos proveedores (<40) y
  procesos (20-50), coherente con mercados locales/nicho mas que con anomalias de dato.
  <mark>Actualizacion (`Capacidad1_cierre_final.ipynb`, seccion 2):</mark> el caso
  Ministerio de Minas y Energia queda explicado — es el mayor de los 109 "fondos
  administrados" identificados alli (contrato con GECELCA S.A. E.S.P., dic-2025,
  ~4.4 billones reales, un unico proveedor por diseno). No es una rareza sin resolver:
  es un patron legitimo de administracion de fondos en el sector energetico/publico, y
  como tal debe reportarse como categoria aparte del "mercado competido", no como un caso
  de concentracion problematica a corregir.
- **INVIAS sigue siendo la entidad mas competitiva** (HHI 163.98, 656 proveedores
  distintos, 972 procesos) tanto antes como despues de la correccion — es un buen caso de
  referencia de "mercado sano" para contrastar en el reporte contra los casos concentrados.
- Conclusion para el reporte: **el mercado CTeI colombiano es competitivo a nivel
  agregado, pero con concentracion real y no trivial en entidades/sectores especificos**
  (energia, licores/alcoholes departamentales, SENA regional) — el mensaje correcto no es
  "hay monopolio" (falso, era el outlier) ni "todo es perfectamente competido" (tampoco
  es cierto, hay nichos concentrados de forma legitima, varios de ellos explicados por
  fondos administrados de un solo operador).

# **C. Picos de enero: ¿estacionalidad real o artefacto?**

Enero 2022 (23,499) coincide con el borde del filtro de fecha (>= 2022-01-01), y
enero 2026 (30,298) duplica cualquier mes normal. Se revisa la distribucion POR DIA
dentro de esos meses: si la masa esta concentrada en el dia 1, es un artefacto de
carga/registro masivo, no comportamiento real de publicacion.

In [8]:
df_fechas = df_proc[df_proc["fecha_de_publicacion_del"].notna()].copy()

for anio in [2022, 2023, 2024, 2025, 2026]:
    enero = df_fechas[
        (df_fechas["fecha_de_publicacion_del"].dt.year == anio)
        & (df_fechas["fecha_de_publicacion_del"].dt.month == 1)
    ]
    if len(enero) == 0: continue
    por_dia = enero["fecha_de_publicacion_del"].dt.day.value_counts().sort_index()
    dia_max = por_dia.idxmax()
    print(f"Enero {anio}: {len(enero):>6,} procesos | dia con mas procesos: {dia_max} "
          f"({por_dia.max():,} = {por_dia.max()/len(enero):.0%} del mes)")

Enero 2022: 23,499 procesos | dia con mas procesos: 28 (2,348 = 10% del mes)
Enero 2023: 11,549 procesos | dia con mas procesos: 31 (920 = 8% del mes)
Enero 2024:  9,602 procesos | dia con mas procesos: 31 (739 = 8% del mes)
Enero 2025: 11,781 procesos | dia con mas procesos: 17 (782 = 7% del mes)
Enero 2026: 30,298 procesos | dia con mas procesos: 30 (2,541 = 8% del mes)


In [9]:
# Vista grafica de los eneros anomalos vs uno normal
fig = make_subplots(rows=1, cols=3, subplot_titles=["Enero 2022", "Enero 2024 (normal)", "Enero 2026"])
for j, anio in enumerate([2022, 2024, 2026], start=1):
    enero = df_fechas[
        (df_fechas["fecha_de_publicacion_del"].dt.year == anio)
        & (df_fechas["fecha_de_publicacion_del"].dt.month == 1)
    ]
    por_dia = enero["fecha_de_publicacion_del"].dt.day.value_counts().sort_index()
    fig.add_trace(go.Bar(x=por_dia.index, y=por_dia.values, showlegend=False), row=1, col=j)
fig.update_layout(title="Procesos por dia del mes en eneros seleccionados", template="plotly_white")
fig.show()

In [10]:
# Estacionalidad presentable: solo anios completos y estables (2023-2025)
df_estable = df_fechas[df_fechas["fecha_de_publicacion_del"].dt.year.isin([2023, 2024, 2025])]
orden_meses = ["Enero","Febrero","Marzo","Abril","Mayo","Junio",
               "Julio","Agosto","Septiembre","Octubre","Noviembre","Diciembre"]
por_mes = (
    df_estable["fecha_de_publicacion_del"].dt.month
    .map(dict(enumerate(orden_meses, start=1)))
    .value_counts().reindex(orden_meses)
)
fig = go.Figure(go.Bar(x=por_mes.index, y=por_mes.values))
fig.update_layout(title="Estacionalidad mensual 2023-2025 (anios completos y estables)",
                   template="plotly_white")
fig.show()

## C.1 Hallazgos — estacionalidad

- **Los picos de enero 2022 y 2026 NO son artefactos de carga masiva**: si fueran una
  recarga/duplicacion concentrada en un solo dia, se veria un dia acumulando una fraccion
  enorme del mes. No es el caso — el dia con mas procesos nunca supera el 10% del mes en
  ningun enero (2022: dia 28 con 10%; 2023: dia 31 con 8%; 2024: dia 31 con 8%; 2025: dia
  17 con 7%; 2026: dia 30 con 8%), practicamente el mismo nivel de concentracion diaria
  que los eneros "normales" 2023-2025. **Esto revierte la sospecha planteada en el EDA
  anterior**: el volumen extra de enero 2022 y 2026 esta distribuido a lo largo del mes,
  consistente con actividad real de publicacion, no con un evento de carga puntual.
- Dicho esto, la magnitud si sigue siendo atipica y queda sin explicar del todo: enero
  2026 (30,298) es ~2.6x el promedio de enero 2023-2025 (~11,270) y enero 2022 (23,499)
  es ~2.1x ese mismo promedio. Como no es un artefacto de un solo dia, hay dos hipotesis
  reales que valdria la pena documentar como limitacion (no se puede resolver solo con
  este dataset): (a) 2022 es el primer anio del dataset y puede incluir una acumulacion
  de procesos "atrasados" que se publicaron todos al arrancar el tracking, y (b) 2026 es
  el anio mas reciente y podria reflejar un cambio real de politica/ciclo presupuestal o
  mayor digitalizacion de entidades — no hay forma de distinguir ambas con los datos
  disponibles.
- **La estacionalidad de referencia queda bien establecida usando solo 2023-2025**: son
  3 anios completos y con volumen mensual estable (sin el sesgo de arranque de 2022 ni el
  corte incompleto de 2026), la base correcta para cualquier afirmacion de "estacionalidad
  tipica" en el reporte.

# **D. Diseno de modelado — Capacidad 3 (decisiones documentables)**

## D.1 Dos universos de modelado, no uno

| Tarea del reto | Universo | Justificacion |
|---|---|---|
| Probabilidad de adjudicacion | Solo modalidades **competitivas** | En regimen especial y contratacion directa (82% del dataset) el campo `adjudicado` es estructuralmente 0: no hay senal que aprender |
| Tipo de contratacion futura, rangos de presupuesto, sectores con mayor inversion | **Todos** los procesos | Estas predicciones no dependen del campo adjudicado |

In [11]:
MODALIDADES_COMPETITIVAS = [
    "Licitación pública", "Licitación pública Obra Publica",
    "Licitación Pública Acuerdo Marco de Precios",
    "Concurso de méritos abierto", "Concurso de méritos con precalificación",
    "Selección Abreviada de Menor Cuantía",
    "Seleccion Abreviada Menor Cuantia Sin Manifestacion Interes",
    "Selección abreviada subasta inversa", "Mínima cuantía",
    "Contratación Directa (con ofertas)", "Contratación régimen especial (con ofertas)",
]
df_competitivo = df_proc[df_proc["modalidad_de_contratacion"].isin(MODALIDADES_COMPETITIVAS)].copy()
print(f"Universo competitivo: {len(df_competitivo):,} procesos "
      f"({len(df_competitivo)/len(df_proc):.1%} del total)")
print(f"Tasa de adjudicacion en el universo competitivo: "
      f"{df_competitivo['adjudicado_proceso'].mean():.1%}")
print()
print(df_competitivo.groupby("modalidad_de_contratacion")["adjudicado_proceso"]
      .agg(tasa="mean", n="size").assign(tasa=lambda d: (d["tasa"]*100).round(1))
      .sort_values("n", ascending=False))

Universo competitivo: 72,236 procesos (14.7% del total)
Tasa de adjudicacion en el universo competitivo: 53.8%

                                                    tasa      n
modalidad_de_contratacion                                      
Mínima cuantía                                     77.70  22685
Concurso de méritos abierto                        39.90  16755
Selección Abreviada de Menor Cuantía                9.80  10464
Contratación Directa (con ofertas)                 78.40   8210
Contratación régimen especial (con ofertas)        66.40   5686
Selección abreviada subasta inversa                40.10   4909
Licitación pública                                 39.60   2134
Licitación pública Obra Publica                    39.50   1191
Seleccion Abreviada Menor Cuantia Sin Manifesta... 37.20    156
Concurso de méritos con precalificación            20.50     39
Licitación Pública Acuerdo Marco de Precios        42.90      7


## D.2 Clasificacion de features por momento de disponibilidad (control de fuga)

Momento de prediccion definido: **al publicarse el proceso**.

| Feature | ¿Disponible al publicar? | Uso |
|---|---|---|
| segmento/familia UNSPSC, modalidad, tipo_de_contrato, entidad, territorio, precio_base, duracion, numero_de_lotes, mes/anio de publicacion, texto del procedimiento | Si | Feature valida |
| historial agregado de la entidad/proveedor ANTES de la fecha del proceso (tasas pasadas, HHI pasado) | Si, con ventana temporal correcta | Feature valida (calcular solo con datos previos) |
| respuestas_al_procedimiento, conteo_de_respuestas_a_ofertas | NO (se conocen al cierre) | Excluir del modelo de adjudicacion |
| proveedores_con_invitacion, proveedores_unicos_con | Parcial (depende de la modalidad) | Documentar y decidir por modalidad |
| valor_total_adjudicacion, fecha_adjudicacion, nombre/nit del proveedor adjudicado | NO (son el resultado) | Solo como target o para features historicas |

In [12]:
# Verificacion rapida del split temporal sobre el universo competitivo
df_comp_fecha = df_competitivo[df_competitivo["fecha_de_publicacion_del"].notna()].copy()
FECHA_CORTE_SPLIT = "2025-07-01"
train = df_comp_fecha[df_comp_fecha["fecha_de_publicacion_del"] < FECHA_CORTE_SPLIT]
test = df_comp_fecha[df_comp_fecha["fecha_de_publicacion_del"] >= FECHA_CORTE_SPLIT]
print(f"Universo competitivo — Train: {len(train):,} | Test: {len(test):,}")
print(f"Tasa adjudicacion train: {train['adjudicado_proceso'].mean():.1%} | "
      f"test: {test['adjudicado_proceso'].mean():.1%}")

Universo competitivo — Train: 52,043 | Test: 17,641
Tasa adjudicacion train: 55.4% | test: 55.1%


##### Deflactar montos COP a pesos constantes usando el IPC del DANE

**Por que:** comparar valor_total_adjudicacion o precio_base entre 2022 y 2026
en pesos NOMINALES mezcla dos efectos (inflacion + cambio real). Para
comparar niveles de valor entre años/meses distintos hay que llevarlos a
la misma base ("pesos constantes de <periodo base>").

Que SI necesita esto: series de valor por mes/año, tamaño promedio de
contrato en el tiempo, "rango de presupuesto" como feature para la
capacidad 3 del reto.

Que NO necesita esto: HHI, participacion de mercado (%), tasas de
adjudicacion, distribucion por sector — son razones calculadas DENTRO de
un mismo periodo, la inflacion se cancela entre numerador y denominador.

Fuente del IPC: DANE, serie historica nacional (Total IPC, no "sin alimentos").
Como esa pestana no trae Indice mensual detallado, el archivo por defecto
(`ipc_dane_mensual_interpolado_TOTAL.csv`) se construyo interpolando
geometricamente entre 10 anclas semestrales confirmadas contra comunicados
oficiales del DANE (dic-2021 a jun-2026): 4 cierres de diciembre (variacion
anual oficial) + 5 junios (variacion "ano corrido" sobre el diciembre
anterior). Julio 2026 no tiene ancla propia todavia (el DANE publica el
IPC el 5to dia habil del mes siguiente) asi que queda igual a junio 2026 —
tratarlo como el limite de precision de esta version, no como un mes real
medido. Si en algun momento consigues el Indice mensual detallado y
verificado de la pestana "Total nacional" (no "sin alimentos"), reemplaza
este archivo por ese y la precision mejora de semestral-interpolada a
mensual-real sin tocar el resto del codigo.

In [ ]:
RUTA_IPC = Path("ipc_dane_mensual_interpolado_TOTAL.csv")
BASE = "2026-06"  # ultima ancla real; julio queda igual (ver nota arriba)


def cargar_ipc(path: Path = RUTA_IPC) -> pd.DataFrame:
    ipc = pd.read_csv(path)
    ipc["anio_mes"] = pd.to_datetime(
        ipc["anio"].astype(str) + "-" + ipc["mes"].astype(str).str.zfill(2) + "-01"
    )
    return ipc[["anio_mes", "indice"]].sort_values("anio_mes").reset_index(drop=True)


def construir_deflactor(ipc: pd.DataFrame, base: str = BASE) -> pd.DataFrame:
    """factor_deflactor: multiplicar un valor nominal de ese mes por este
    factor para expresarlo en pesos del mes `base`.
    factor = indice_base / indice_del_mes
    """
    indice_base = ipc.loc[ipc["anio_mes"] == pd.Timestamp(base + "-01"), "indice"]
    if indice_base.empty:
        raise ValueError(f"No hay indice IPC para el periodo base {base}. "
                          f"Verifica que la serie descargada llegue hasta ahi.")
    indice_base = indice_base.iloc[0]
    out = ipc.copy()
    out["factor_deflactor"] = indice_base / out["indice"]
    return out


def deflactar(df: pd.DataFrame, col_fecha: str, columnas_monetarias: list[str],
              deflactor: pd.DataFrame) -> pd.DataFrame:
    """Agrega columnas `<col>_real` en pesos constantes del periodo BASE.

    Left-merge por mes: filas sin fecha valida quedan con NaN en las
    columnas _real (coherente con el resto del pipeline, que ya trata la
    fecha faltante como informativa, no como dato a imputar).
    """
    out = df.copy()
    out["_anio_mes"] = out[col_fecha].dt.to_period("M").dt.to_timestamp()
    out = out.merge(
        deflactor[["anio_mes", "factor_deflactor"]],
        left_on="_anio_mes", right_on="anio_mes", how="left"
    )
    for col in columnas_monetarias:
        out[f"{col}_real"] = out[col] * out["factor_deflactor"]
    return out.drop(columns=["_anio_mes", "anio_mes"])


if __name__ == "__main__":
    ipc = cargar_ipc()
    deflactor = construir_deflactor(ipc)
    print(deflactor.tail())

    df_proc = pd.read_csv("secop_ctei_procesos_limpio.csv")
    df_proc["fecha_de_publicacion_del"] = pd.to_datetime(df_proc["fecha_de_publicacion_del"], errors="coerce")

    df_proc = deflactar(df_proc, "fecha_de_publicacion_del",
                         ["precio_base", "valor_adjudicado_total"], deflactor)

    print(df_proc[["fecha_de_publicacion_del", "valor_adjudicado_total",
                    "valor_adjudicado_total_real"]].head())

    df_proc.to_csv("secop_ctei_procesos_deflactado.csv", index=False, encoding="utf-8-sig")
    print("guardado: secop_ctei_procesos_deflactado.csv")

     anio_mes  indice  factor_deflactor
50 2026-03-01  139.86              1.02
51 2026-04-01  140.98              1.02
52 2026-05-01  142.07              1.01
53 2026-06-01  143.20              1.00
54 2026-07-01  143.20              1.00
  fecha_de_publicacion_del  valor_adjudicado_total  \
0               2026-01-28                       0   
1               2026-01-30                       0   
2               2026-01-28                       0   
3               2026-01-28                       0   
4               2026-01-28                       0   

   valor_adjudicado_total_real  
0                         0.00  
1                         0.00  
2                         0.00  
3                         0.00  
4                         0.00  
guardado: secop_ctei_procesos_deflactado.csv


## D.3 Hallazgos — diseno de modelado

- **Restringir al universo competitivo arregla el desbalance casi por completo**: 72,236
  procesos (14.7% del dataset total) con tasa de adjudicacion de **53.8%**, muy lejos del
  7.89% global enganoso del EDA original. Esto confirma el diagnostico: el desbalance
  extremo de 4.5/4.1 del primer notebook era, en su mayor parte, un artefacto de mezclar
  modalidades donde el target no aplica. El problema de "clase minoritaria" para el
  modelo de capacidad 3 es mucho mas leve de lo que parecia.
- El split temporal (mismo corte 2025-07-01) sobre el universo competitivo queda muy
  balanceado: train 52,043 (55.4%) vs. test 17,641 (55.1%) — practicamente identico, la
  mejor senal posible de que el split no introduce sesgo de proporciones.
- La tabla de disponibilidad de features (D.2) deja trazabilidad explicita del control de
  fuga: confirma que `respuestas_al_procedimiento` (la variable con mayor correlacion en
  el EDA, 0.78) se descarta correctamente por conocerse solo al cierre. Queda pendiente
  una decision explicita (no solo "documentar") sobre `proveedores_con_invitacion` y
  similares: como su disponibilidad depende de la modalidad, conviene resolverla por
  modalidad en el notebook de modelado, no dejarla ambigua.
- **Gap detectado en el script de deflactacion**: `deflactar()` se aplica sobre
  `secop_ctei_procesos_limpio.csv` leido de nuevo desde cero (variable `df_proc` local al
  bloque `__main__`), **sin aplicar el filtro `flag_valor_implausible`** definido en la
  seccion A. Si el proceso `CO1.REQ.6849316` (EAG) aporta su valor de linea al agregado
  `valor_adjudicado_total` de `df_proc`, el archivo de salida
  `secop_ctei_procesos_deflactado.csv` hereda el mismo outlier de magnitud imposible,
  ahora ademas multiplicado por un `factor_deflactor` — hay que verificar esto en el
  notebook de modelado antes de usar las columnas `_real` para cualquier feature o
  grafico, o refiltrar por plausibilidad antes de deflactar.
- El diseno de deflactacion en si (IPC interpolado semestral, base jun-2026, julio-2026
  sin ancla propia) es metodologicamente razonable y bien documentado; la limitacion de
  precision (mensual real vs. semestral interpolado) esta explicitada en el propio codigo,
  buena practica a mantener.

# **E. Conclusiones para el reporte**

1. **Dataset verificado y confiable en estructura** (cuadre lineas/procesos, nulos
   interpretables), pero **requiere el filtro de plausibilidad monetaria antes de
   cualquier metrica de valor** — sin el, el HHI y el ranking de mercado quedan
   completamente invalidados por 14 lineas (0.003% de los datos).
2. **Mercado CTeI colombiano: competitivo a nivel agregado (HHI 116, 6.1% de
   proveedores concentran el 80% del valor), con nichos reales de concentracion** en
   energia, licores/alcoholes departamentales y SENA regional — mensaje matizado, no
   "hay monopolio" ni "todo es perfectamente competido".
3. **Estacionalidad**: enero domina por ciclo presupuestal, confirmado con datos
   distribuidos a lo largo del mes (no artefacto de carga); referencia estable = 2023-2025.
   Los picos de 2022 y 2026 son reales pero de causa no determinable con este dataset —
   documentar como limitacion, no como error a corregir.
4. **Modelado de capacidad 3 debe restringirse al universo competitivo** (72,236
   procesos, 14.7% del total, 53.8% de tasa de adjudicacion) con control de fuga
   explicito (excluir `respuestas_al_procedimiento` y variables post-cierre) y el mismo
   corte temporal 2025-07-01, que queda bien balanceado en ese universo.
5. **Pendiente antes de entrenar cualquier modelo**: resolver el gap de la deflactacion
   (filtrar `flag_valor_implausible` antes de deflactar) y decidir el tratamiento de
   `proveedores_con_invitacion` por modalidad.